# Module 3: Gold layer, a star schema and analysis tables

**Input:** `workspace.dubai_re.silver_transactions`
**Output:** seven gold tables:

| Table | Type | Rows (approx.) | Used by |
|-------|------|----------------|---------|
| `fact_sales` | Fact | 1,015,297 | Power BI, Q2 |
| `dim_date` | Dimension | 4,383 | Power BI, all |
| `dim_location` | Dimension | 341 | Power BI, Q1 |
| `dim_property` | Dimension | 74 | Power BI, Q1, Q2 |
| `agg_market_monthly` | Aggregate | 142 | Q5 forecast, Excel |
| `agg_price_community_yearly` | Aggregate | ~5,700 | Q1 hotspots, Excel |

**Time:** about 2 hours. New skills: dimensional modelling, surrogate keys, generating a date table, anti joins for integrity testing, multi-table SQL joins, and exporting results for Excel and Python.

## 🧠 Why we're doing this: the star schema

Silver is one wide table: every row repeats `"Jumeirah Village Circle"`, `"Residential"`, `"Flat"`… That's fine for Spark, but BI tools (Power BI especially) work best with a **star schema**:

```
                 ┌──────────────┐
                 │   dim_date   │  one row per day
                 └──────┬───────┘
                        │ date_key
┌──────────────┐  ┌─────┴──────┐  ┌──────────────┐
│ dim_location ├──┤ fact_sales ├──┤ dim_property │
└──────────────┘  └────────────┘  └──────────────┘
 one row per        one row per      one row per type of
 community/area     core sale        property (type, usage, beds)
```

- **Fact table:** the *events* you measure (sales), holding numbers (price, area) and **keys** pointing to the dimensions. It's long and narrow.
- **Dimension tables:** the *descriptions* you filter and group by (when, where, what). They're short and wide.
- **Grain:** what one fact row represents. Here, *one row = one core sale registered from 2015 onwards*. Always state the grain first. Most modelling mistakes come from mixing grains.

**You already know this from Excel:** the fact table is the transactions sheet, and the dimensions are lookup sheets. Instead of `XLOOKUP`-ing community details into every row, Power BI *relates* the tables through the keys.

**Why the 2015 start?** It gives almost 12 years: enough for seasonality and the post-2020 cycle, without old years where volumes were small. It's written down as `ANALYSIS_START`, so it's a visible, changeable decision.

🎤 **Interview angle:** *"Walk me through your data model."* → *"A star schema with one fact table at the grain of one sale, and date, location and property dimensions joined on integer surrogate keys. That's the shape Power BI's engine is optimised for."*

## Setup

In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "dubai_re"
T = lambda name: f"{CATALOG}.{SCHEMA}.{name}"      # T("fact_sales") -> "workspace.dubai_re.fact_sales"

SILVER_TABLE = T("silver_transactions")
ANALYSIS_START = "2015-01-01"

def check(label, actual, expected):
    fmt = (lambda v: f"{v:,}") if isinstance(expected, int) else str
    if actual == expected:
        print(f"✅ {label}: {fmt(actual)}")
    else:
        print(f"❌ {label}: got {actual}, expected {fmt(expected)}. Re-read the hint above this cell.")

def check_close(label, actual, expected, tolerance=0.02):
    if actual is None:
        print(f"❌ {label}: got nothing back (None). Check the cell above.")
        return
    ok = abs(actual - expected) <= tolerance * expected
    print(f"{'✅' if ok else '❌'} {label}: got {actual:,.0f}, expected about {expected:,.0f} (±{tolerance:.0%})")

## Step 1: Select the sales in scope and find where the data ends

🧠 **Why we're doing this**
Gold is built for questions, so we keep only what the questions need: **core sales** with a **valid date** from **2015 onwards**.

We also *measure* the last date in the data instead of typing `2026-11-05` into the code. When you download a fresh extract next month, everything adjusts automatically. Avoiding hard-coded values like this is called **parameterising**.

✏️ **TODO 1a:** filter silver to core sales, valid dates, and `transaction_date >= ANALYSIS_START`.
*Hint:* chain three `.filter()` calls, or combine the conditions with `&` in one.
✏️ **TODO 1b:** get the latest `transaction_date` in `df_core` with `F.max`.

In [ ]:
silver = spark.table(SILVER_TABLE)

# TODO 1a: core sales, valid dates, from ANALYSIS_START
df_core = (
    silver
      .filter(F.col("is_core_sale"))
      .filter(F.col("is_valid_date"))
      .filter(F.col("transaction_date") >= ANALYSIS_START)
)

# TODO 1b: the last date in the data
DATA_END = df_core.agg(F.max("transaction_date").alias("max_date")).first()["max_date"]

print("Data ends on:", DATA_END)
check("Core sales in scope", df_core.count(), 1_015_297)
check("Last transaction date", str(DATA_END), "2026-11-05")

## Step 2: Build the date dimension (in SQL)

🧠 **Why we're doing this**
A **date table** has one row for *every* day, including days with no sales. It's the backbone of time analysis. Power BI's time intelligence (year-on-year, year-to-date) needs one, and it guarantees that a month with zero sales still shows up as a zero instead of disappearing from a chart.

We *generate* the days instead of reading them:
- `sequence(start, end, INTERVAL 1 DAY)` makes an **array** of every date.
- `explode()` turns one array into one row per element, like *unpivoting* a list into rows.

`is_complete_month` marks months that ended before the data did. November 2026 only runs to the 5th, so it must never be compared with a full month. That one flag protects every monthly chart and the forecast.

`date_key` is the date as an integer (`20251031`). Integer keys make joins and Power BI relationships fast, and they're still readable.

✏️ **TODO 2:** complete the three `___` columns: `YEAR(d)`, `MONTH(d)`, and the month start `trunc(d, 'MM')`.

In [ ]:
DIM_DATE = T("dim_date")
date_end = f"{DATA_END.year}-12-31"     # through the end of the last year in the data

spark.sql(f"""
CREATE OR REPLACE TABLE {DIM_DATE} AS
SELECT
    CAST(date_format(d, 'yyyyMMdd') AS INT)   AS date_key,
    d                                         AS date,
    YEAR(d)                                   AS year,          -- TODO 2
    QUARTER(d)                                AS quarter,
    MONTH(d)                                  AS month,         -- TODO 2
    date_format(d, 'MMM')                     AS month_name,
    trunc(d, 'MM')                            AS month_start,   -- TODO 2
    date_format(d, 'yyyy-MM')                 AS year_month,
    last_day(d) <= DATE'{DATA_END}'           AS is_complete_month
FROM (
    SELECT explode(sequence(DATE'{ANALYSIS_START}', DATE'{date_end}', INTERVAL 1 DAY)) AS d
)
""")

dd = spark.table(DIM_DATE)
check("Days in dim_date", dd.count(), 4_383)
check("Complete months", dd.filter("is_complete_month").select("month_start").distinct().count(), 142)
display(dd.orderBy("date").limit(5))

## Step 3: Business categories, segment and bedroom band

🧠 **Why we're doing this**
Silver has 74 combinations of property type × sub-type × usage × bedrooms, far too many for a dashboard slicer. The business thinks in **segments** (apartments, villas, land…) and **bedroom bands**. We define those once, here, so Power BI, SQL and Python all use exactly the same groups.

**Order matters in a `when` chain.** The first matching condition wins, just like nested `IF`s in Excel. Land and buildings are tested first, so later rules can't accidentally catch them.

`bedroom_sort` is a number for ordering. Power BI would otherwise sort bands alphabetically (`1 BR, 2 BR, 5+ BR, N/A, Studio`). In Power BI you'll use **Sort by column → bedroom_sort**.

✏️ **TODO 3:** complete the Apartment rule: property type `Unit`, sub-type `Flat`, usage `Residential`.
*Hint:* three conditions joined with `&`, each in its own brackets.

In [ ]:
segment = (
    F.when(F.col("property_type") == "Land", "Land")
     .when(F.col("property_type") == "Building", "Building")
     # TODO 3: the Apartment rule
     .when((F.col("property_type") == "Unit") &
           (F.col("property_sub_type") == "Flat") &
           (F.col("property_usage") == "Residential"), "Apartment")
     .when((F.col("property_usage") == "Residential") &
           ((F.col("property_type") == "Villa") | (F.col("property_sub_type") == "Stacked Townhouses")),
           "Villa / Townhouse")
     .when((F.col("property_type") == "Unit") & (F.col("property_usage") == "Hospitality"), "Hotel apartment")
     .when(F.col("property_sub_type") == "Office", "Office")
     .when(F.col("property_sub_type").isin("Shop", "Show Rooms"), "Retail")
     .otherwise("Other")
)

bedroom_band = (
    F.when(F.col("bedrooms").isNull(), "N/A")
     .when(F.col("bedrooms") == 0, "Studio")
     .when(F.col("bedrooms") >= 5, "5+ BR")
     .otherwise(F.concat(F.col("bedrooms").cast("string"), F.lit(" BR")))
)

bedroom_sort = F.when(F.col("bedrooms").isNull(), 99).otherwise(F.least(F.col("bedrooms"), F.lit(5)))

df_core = (
    df_core
      .withColumn("segment", segment)
      .withColumn("bedroom_band", bedroom_band)
      .withColumn("bedroom_sort", bedroom_sort)
)

seg = {r["segment"]: r["count"] for r in df_core.groupBy("segment").count().collect()}
check("Apartment sales", seg.get("Apartment"), 723_633)
check("Villa / Townhouse sales", seg.get("Villa / Townhouse"), 113_178)
check("Segments", len(seg), 8)

## Step 4: Surrogate keys

🧠 **Why we're doing this**
The fact table needs a single column that points to each dimension row. A location is identified by *two* text columns (`community` + `dld_area`), and a property type by *four*. Power BI relationships can only use **one column**, and joining on long text is slow. So we create a **surrogate key**: a single integer that stands for the combination.

We make it with a **hash**: `xxhash64` turns any text into a 64-bit integer, and the same input always gives the same number. Compared with numbering rows 1, 2, 3…, hashing has two advantages:
- **Deterministic:** re-running the notebook gives every location the same key, so nothing downstream breaks.
- **No lookup needed:** the fact table can compute its keys on its own, without joining to the dimension first.

**The NULL trap.** Spark's hash functions *skip* NULLs, so `(A, NULL)` and `(NULL, A)` could get the same key. The helper replaces NULL with a marker and joins the values with `||` before hashing, so every combination stays distinct.

The helper is written for you. ✏️ **TODO 4:** add `property_key` using `make_key(*PROPERTY_COLS)`. The `*` unpacks the list into separate arguments.

In [ ]:
def make_key(*cols):
    """Deterministic 64-bit surrogate key from one or more columns (NULL-safe)."""
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in cols]
    return F.xxhash64(F.concat_ws("||", *parts))

LOCATION_COLS = ["community", "dld_area"]
PROPERTY_COLS = ["property_type", "property_sub_type", "property_usage", "bedrooms"]

df_core = (
    df_core
      .withColumn("location_key", make_key(*LOCATION_COLS))
      # TODO 4: the property key
      .withColumn("property_key", make_key(*PROPERTY_COLS))
      .withColumn("date_key", F.date_format("transaction_date", "yyyyMMdd").cast("int"))
)

display(df_core.select("transaction_id", "date_key", "location_key", "community", "property_key", "segment", "bedroom_band").limit(5))

## Step 5: Build the location and property dimensions

🧠 **Why we're doing this**
A dimension is the **distinct** set of descriptions, one row per key. `select(...).distinct()` is `SELECT DISTINCT` in SQL, or **Remove Duplicates** in Excel.

The check that matters most: **the number of rows must equal the number of distinct keys.** If it doesn't, one key points to two descriptions, and Power BI would refuse the relationship (or, worse, double count).

✏️ **TODO 5:** build `dim_property` the same way as `dim_location`: `property_key`, the four `PROPERTY_COLS`, plus `segment`, `bedroom_band` and `bedroom_sort`, then `.distinct()`.

In [ ]:
dim_location = df_core.select("location_key", *LOCATION_COLS).distinct()

# TODO 5: the property dimension
dim_property = df_core.select("property_key", *PROPERTY_COLS, "segment", "bedroom_band", "bedroom_sort").distinct()

check("dim_location rows", dim_location.count(), 341)
check("dim_location unique keys", dim_location.select("location_key").distinct().count(), 341)
check("dim_property rows", dim_property.count(), 74)
check("dim_property unique keys", dim_property.select("property_key").distinct().count(), 74)

🤔 **Think about it:** there are 318 distinct community names but 341 rows in `dim_location`. What does that tell you about how communities and DLD areas relate? (Hint: one community can span more than one DLD land area.)

**Answer:** some master communities sit across several official DLD land areas. For example, a large master project can cover two or three numbered DLD areas. So `community` alone isn't unique in the dimension, but `community + dld_area` is. That's why the key uses both. When Power BI groups by `community`, it correctly adds up all of that community's areas.

## Step 6: Build the fact table and save all four tables

🧠 **Why we're doing this**
The fact table keeps only **keys, measures and a few transaction-level details**. Descriptions like `community` or `segment` live in the dimensions, which is what keeps the fact table narrow and fast.

`market` (`Off-plan` / `Ready`) stays on the fact as a readable label. A two-value attribute that belongs to the transaction itself doesn't need its own dimension. `project` and `building` also stay on the fact (they're too detailed for `dim_location`), so you can still drill down to a single tower in Power BI.

We save the three PySpark tables in **one loop over a dictionary** instead of three copied blocks. Less code, and no chance of one table being saved with different settings.

✏️ **TODO 6:** in the loop, write each DataFrame as Delta with mode `"overwrite"` to the table name `name`.

In [ ]:
fact_sales = df_core.select(
    "transaction_id",
    "date_key", "location_key", "property_key",
    "is_off_plan",
    F.when(F.col("is_off_plan"), "Off-plan").otherwise("Ready").alias("market"),
    "price_aed", "area_sqft", "price_per_sqft", "is_price_outlier", "has_parking",
    "project", "building",
)

GOLD_TABLES = {
    T("dim_location"): dim_location,
    T("dim_property"): dim_property,
    T("fact_sales"):   fact_sales,
}

for name, df in GOLD_TABLES.items():
    # TODO 6: save each table as Delta (overwrite)
    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(name))
    print(f"saved {name}")

check("fact_sales rows", spark.table(T("fact_sales")).count(), 1_015_297)

## Step 7: Star schema integrity gate

🧠 **Why we're doing this**
A star schema has two promises, and we test both:
1. **Every dimension key is unique** (one description per key).
2. **Every fact key exists in its dimension**, which is called **referential integrity**. A fact row whose key is missing from the dimension is an *orphan*. In Power BI it lands in a "(Blank)" bucket and quietly disappears from filtered visuals.

**Anti join.** `fact.join(dim, "key", "left_anti")` returns fact rows with **no match** in the dimension. In SQL it's `WHERE NOT EXISTS (…)`, or a `LEFT JOIN … WHERE dim.key IS NULL`. In Excel, it's the rows where your `XLOOKUP` returns `#N/A`. We want zero.

✏️ **TODO 7:** add the orphan test for `property_key` against `dim_property`, following the pattern of the location test.

In [ ]:
f  = spark.table(T("fact_sales"))
dl = spark.table(T("dim_location"))
dp = spark.table(T("dim_property"))
dd = spark.table(T("dim_date"))

tests = {
    "fact transaction_id unique":       f.select("transaction_id").distinct().count() == f.count(),
    "dim_location keys unique":         dl.select("location_key").distinct().count() == dl.count(),
    "dim_property keys unique":         dp.select("property_key").distinct().count() == dp.count(),
    "dim_date keys unique":             dd.select("date_key").distinct().count() == dd.count(),
    "no orphan location_key":           f.join(dl, "location_key", "left_anti").count() == 0,
    # TODO 7: no orphan property_key
    "no orphan property_key":           f.join(dp, "property_key", "left_anti").count() == 0,
    "no orphan date_key":               f.join(dd, "date_key", "left_anti").count() == 0,
    "no NULL measures":                 f.filter("price_aed IS NULL OR price_per_sqft IS NULL").count() == 0,
}

for name, passed in tests.items():
    print(f"{'✅' if passed else '❌'} {name}")

failed = [n for n, p in tests.items() if not p]
if failed:
    raise ValueError(f"Star schema integrity failed: {failed}")
print("\n🎉 Star schema passed all integrity tests.")

## Step 8: Aggregate table for Q5, the monthly market

🧠 **Why we're doing this**
The forecast in Module 6 needs one row per month: how many sales, what value, the off-plan/ready split, and the typical apartment price. Building it here, in SQL, from the star schema means the Python forecast, the Excel summary and the Power BI cards all use **exactly the same numbers**.

New SQL ideas:
- **Joining a star:** fact in the middle, one `JOIN` per dimension on its key.
- **Conditional aggregation:** `SUM(CASE WHEN … THEN 1 ELSE 0 END)` counts only the rows that match. That's the SQL version of `COUNTIFS`.
- **`MEDIAN(CASE WHEN … THEN x END)`:** rows that don't match become NULL, and `MEDIAN` ignores NULLs. So you get the apartment-only median in the same query.
- **`WHERE d.is_complete_month`:** the partial November is excluded once, here.

Nothing to fill in for this one. Read it line by line, run it, and make sure you could explain each line in an interview.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("agg_market_monthly")} AS
SELECT
    d.month_start,
    COUNT(*)                                                  AS sales,
    ROUND(SUM(f.price_aed) / 1e9, 2)                          AS value_aed_bn,
    SUM(CASE WHEN f.is_off_plan     THEN 1 ELSE 0 END)        AS off_plan_sales,
    SUM(CASE WHEN NOT f.is_off_plan THEN 1 ELSE 0 END)        AS ready_sales,
    ROUND(MEDIAN(CASE WHEN p.segment = 'Apartment' AND NOT f.is_price_outlier
                      THEN f.price_per_sqft END))             AS median_apartment_aed_sqft
FROM {T("fact_sales")}   f
JOIN {T("dim_date")}     d ON f.date_key     = d.date_key
JOIN {T("dim_property")} p ON f.property_key = p.property_key
WHERE d.is_complete_month
GROUP BY d.month_start
ORDER BY d.month_start
""")

mm = spark.table(T("agg_market_monthly"))
oct25 = mm.filter("month_start = '2025-10-01'").first()

check("Months", mm.count(), 142)
check("Total sales across complete months", mm.agg(F.sum("sales")).first()[0], 1_012_986)
check("Oct 2025 sales", oct25["sales"], 19_331)
check("Oct 2025 off-plan sales", oct25["off_plan_sales"], 12_952)
check_close("Oct 2025 median apartment AED/sq ft", oct25["median_apartment_aed_sqft"], 1_712, tolerance=0.01)

## Step 9: Aggregate table for Q1, prices by community, segment and bedrooms

🧠 **Why we're doing this**
Q1 asks *where prices are rising fastest*. Module 4 will rank year-on-year changes with window functions. That ranking needs a table at the grain **year × community × segment × bedroom band**, with the median price per sq ft and the number of sales behind it. The sales count matters: a median of 3 sales isn't evidence, so Module 4 will set a minimum.

We keep **apartments and villas/townhouses** (the residential market) and **exclude outliers**. `is_full_year` marks 2026 as year-to-date, so it's never mistaken for a full year.

✏️ **TODO 9:** write the `GROUP BY`. It must list every non-aggregated column in the `SELECT`.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("agg_price_community_yearly")} AS
SELECT
    d.year,
    d.year < {DATA_END.year}                       AS is_full_year,
    l.community,
    p.segment,
    p.bedroom_band,
    p.bedroom_sort,
    COUNT(*)                                       AS sales,
    ROUND(MEDIAN(f.price_per_sqft))                AS median_aed_sqft,
    ROUND(MEDIAN(f.price_aed))                     AS median_price_aed,
    ROUND(AVG(CAST(f.is_off_plan AS INT)) * 100, 1) AS off_plan_pct
FROM {T("fact_sales")}   f
JOIN {T("dim_date")}     d ON f.date_key     = d.date_key
JOIN {T("dim_location")} l ON f.location_key = l.location_key
JOIN {T("dim_property")} p ON f.property_key = p.property_key
WHERE NOT f.is_price_outlier
  AND p.segment IN ('Apartment', 'Villa / Townhouse')
GROUP BY d.year, l.community, p.segment, p.bedroom_band, p.bedroom_sort   -- TODO 9
""")

pc = spark.table(T("agg_price_community_yearly"))
jvc = {r["year"]: r["median_aed_sqft"] for r in pc.filter(
        "community = 'Jumeirah Village Circle' AND segment = 'Apartment' AND bedroom_band = '1 BR'").collect()}

check_close("Rows", pc.count(), 5_702)
check_close("Sales covered", pc.agg(F.sum("sales")).first()[0], 831_646)
check_close("JVC 1-bed apartments, 2025 median AED/sq ft", jvc.get(2025), 1_418, tolerance=0.01)
check_close("JVC 1-bed apartments, 2024 median AED/sq ft", jvc.get(2024), 1_249, tolerance=0.01)

🤔 **Think about it:** `is_full_year` is computed with `d.year < {DATA_END.year}` and not just `d.year < 2026`. Why? And why is `bedroom_sort` in the `GROUP BY` even though it doesn't split any groups?

**Answers**
1. Parameterised, not hard-coded. With next year's extract, 2026 becomes a full year automatically.
2. Every non-aggregated column in the `SELECT` must appear in the `GROUP BY`. `bedroom_sort` is fully determined by `bedroom_band`, so adding it doesn't create extra groups. It just carries the sort order through to Power BI and Excel.

## Step 10: Your first star schema query

🧠 **Why we're doing this**
This is how an analyst uses the model: start at the fact, join the dimensions you need, filter and group by their columns. Practise it here, because Power BI will do exactly this for you behind every visual.

✏️ **TODO 10:** the **top 10 communities by apartment sales in 2025**, with their median AED per sq ft (excluding outliers). The joins are written. Add the `WHERE`, `ORDER BY` and `LIMIT`.

In [ ]:
%sql
-- TODO 10: top 10 communities by apartment sales in 2025
SELECT l.community,
       COUNT(*)                         AS apartment_sales,
       ROUND(MEDIAN(f.price_per_sqft))  AS median_aed_sqft,
       ROUND(AVG(CAST(f.is_off_plan AS INT)) * 100, 1) AS off_plan_pct
FROM workspace.dubai_re.fact_sales f
JOIN workspace.dubai_re.dim_date     d ON f.date_key     = d.date_key
JOIN workspace.dubai_re.dim_location l ON f.location_key = l.location_key
JOIN workspace.dubai_re.dim_property p ON f.property_key = p.property_key
WHERE d.year = 2025
  AND p.segment = 'Apartment'
  AND NOT f.is_price_outlier
GROUP BY l.community
ORDER BY apartment_sales DESC
LIMIT 10

💡 JVC should top the list. Compare its median AED/sq ft with Downtown or Dubai Marina. That gap between *where most people buy* and *where prices are highest* is the kind of insight Q1 and the Power BI dashboard will bring out.

## Step 11: Export the aggregates for Excel and Python

🧠 **Why we're doing this**
Modules 5–7 run in VS Code and Excel on your Mac, so they need files. The two aggregate tables are small (142 and ~5,700 rows), so we bring them into **pandas** with `.toPandas()` and save them as CSV in a new `exports` volume. You then download them from Catalog Explorer.

⚠️ **`.toPandas()` pulls every row into the memory of one machine.** It's fine for a few thousand rows, but never use it on `fact_sales`. Knowing that boundary is a real interview point: *"aggregate in Spark, then move small results to pandas."*

Just run these two cells.

In [ ]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.dubai_re.exports
COMMENT 'Small gold extracts for Excel, Python and Power BI'

In [ ]:
EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/exports/"

for table in ["agg_market_monthly", "agg_price_community_yearly"]:
    pdf = spark.table(T(table)).toPandas()
    pdf.to_csv(EXPORT_PATH + f"{table}.csv", index=False)
    print(f"✅ {table}.csv  ({len(pdf):,} rows)")

display(dbutils.fs.ls(EXPORT_PATH))

**Download them:** go to **Catalog → workspace → dubai_re → Volumes → exports**, click each CSV, then click **Download**. Save them into your repo's `data/gold/` folder (create the folder). They're small, so they *can* go in Git. That means anyone cloning your repo can run the Python and Excel parts without a Databricks account.

## ✅ Module 3 complete

**What you built:** a tested star schema (one fact and three dimensions) plus two aggregate tables, all derived from one set of definitions, with the aggregates exported for Excel and Python.

**🎤 Practise saying these out loud** (model answers in `LEARNING.md`):
1. What is a star schema, and what's the grain of your fact table?
2. What's a surrogate key, and why did you hash instead of numbering rows?
3. Why do you need a date dimension when the fact table already has dates?
4. What is referential integrity, and how did you test it?
5. Fact vs dimension: where would you put `community`, and why?
6. When is `.toPandas()` safe?

**Commit your work**
1. **File → Export → IPython Notebook**, then save over `databricks/03_gold_model.ipynb`
2. Download the two CSVs into `data/gold/`
3. In the VS Code terminal:
```bash
git add databricks/03_gold_model.ipynb data/gold/
git commit -m "Add gold layer: star schema, integrity tests and Q1/Q5 aggregates"
git push
```
4. Tick Module 3 in the README roadmap.

**Next:** Module 4, Q1 price hotspots in Spark SQL. Window functions (`LAG`, `RANK`), year-on-year growth, minimum-volume rules and the first findings for your README.